# HNSW Algorithm Implementation

## **Components**

#### *Search*: method for ANN search

#### *Insertion and Layer Assignment*: building the HNSW structure/inserting nodes for efficient searching



## **Search**

#### Suppose we have n nodes (A-E) in a layer, *query point Q*, and we want the 2 NNs (ef=2). We have the name of each node, their distances to Q, and their graph connections (who is connected to who at this layer). One of the n nodes, A, will be out *entry point*; we start here.

- *min-heap* : For *candidates*, keeps the smallest value at the top, index 0 = smallest dist --> explore next

- *max-heap* : For *found*, keeps the greatest value at the top (as a negative), index 0 = worst result (greatest distance), first to evict

#### We start a loop wherein we:
- (1) Pop the smallest valued node, A, from *candidates*, the node we explore from. 

- (2a) If this value is NOT worse than the *worst_found* (the largest value in *found*), then continue. 

- (2b) If this value is worse than the *worst_found* , discared the node.

#### Assuming (2a), and noting that (i) the first node has been popped out of *candidates* while (ii) staying in *found*, we check out A's neighbors, assuming B and C are designated as such. First we visit B: say the dist(Q,B)=2.0 < *worst_found* (5.0) and push B onto both heaps. 

#### Following (2a) again, we continue to A's other neighbor, C. Suppose dist(Q,C)=8.0 > *worst_found*(5.0). Since *found* is already full, C is discarded entirely.

#### *Now* we start the loop all over again! We pop B from *candidates*, compare with *worst_found* (still A), and explore B's neighbors, which could be [A, D, E]. Since A is already visited, we skip it and visit D.


#### Now that we've done this once in detail, I'll be less verbose from here on out.

#### dist(Q,D)=1.0 < *worst_found* (5.0) --> push D onto both heaps. *found* has 3 items but we pop the worst (A at 5.0) because ef=2.

#### Suppose dist(Q,E)=3.0 < *worst_found* (now 2.0, B). 3.0>2.0, so we discard E.

#### Loop again! Pop smallest from *candidates* (D at 1.0), 1.0 < *worst_found* (2.0), so we explore D's neighbors, say [B, E]. Both visited previously, skip both. Nothing pushed.

#### Candidates now empty, loop ends. Return *found*: [(1.0. D), (2.0, B)]. These are our 2 ANNs.

#### Generally: We consider a node (as a *candidate*), explore its neighbors, and continually update *found* until we have 2 nodes. Then we try continue to the next node, and so on.


In [ ]:

# import heapq

# def search_layer(self, query, entry_points, ef, layer):
#     visited = set(entry_points)
#     candidates = []
#     found = []

#     for ep in entry_points:
#         dist = self.dist(query, self.vectors[ep])
#         heapq.heappush(candidates, (dist, ep)) # push ep to candidates/min-heap w// distance
#         heapq.heappush(found, (-dist, ep)) # push ep to found/max-heap w// distance
    
#     while candidates:
#         c_dist, c_id = heapq.heappop(candidates)
#         worst_found = -found[0][0] # worst distance in found/max-heap
        
#         if c_dist > worst_found:
#             break
        
#         for neighbor_id in self.graphs[layer].get(c_id, []):
#             if neighbor_id in visited: 
#                 continue
#             visited.add(neighbor_id)

#             n_dist = self.dist(query, self.vectors[neighbor_id])
#             worst_found = -found[0][0]

#             if n_dist < worst_found or len(found) < ef:
#                 heapq.heappush(candidates, (n_dist, neighbor_id))
#                 heapq.heappush(found, (-n_dist, neighbor_id))
#                 if len(found) > ef:
#                     heapq.heappop(found)

#     return sorted((-d, nid) for d, nid in found)




## **Select Neighbors**

#### The previous function is used for searching through layers to return a dynamic list of found NNs. This will be used not only for KNN Search but also for HNSW *construction*,  
#### When constructing our multilayer HNSW graph, we also need a means of connecting nodes. There are two methods we consider, the first naive and the second more robust:

#### (i) **Select-Neighbors (simple)**
- #### We want to connect a new node, Q, to neighbors. We use the output of *search_layer* (W, a list of *efConstruction* candidates, e.g. 50 candidates) to select the *M* nodes closest to Q and connect them to Q. Note the difference between *ef* and *efConstruction*; here, we use the latter. 
- #### **Problem:** Suppose we a candidate list sorted by distance to Q (A:1.0, B:1.2, C: 1.4, D:2.0, E:2.5). With parameter *M=3*, we would pick A,B, and C. However, A, B, and C might be clustered in the same region, and our node would end up with neighbors only pointing in one direction. This limits the maneuverability of the graph and ultimately the accuracy of our search.
- ### The heuristic below solves this problem!

#### (ii) **Select-Neighbors (heuristic)**
- #### Instead, a candidate from *M* only earns its place if it is closer to Q than it is to any already-selected neighbor. Assume the distances to Q listed above, but suppose the following are also true: dist(B-->A)=0.2 and dist(C-->A)=3.5. 
- #### Since B is NOT closer to Q than it is to A, it is disqualified. C is selected, however, and is pointing in a different direction.









In [ ]:
# # Select Neighbors (Simple)
# def select_neighbors_simple(self, candidates, M):
#     return [nid for _, nid in candidates[:M]]

# # Select Neighbors (Heuristic)
# def select_neighbors_heuristic(self, query, candidates, M):
#     selected = []
#     for dist_to_query, cid in candidates:
        
#         if len(selected) >= M:
#             break
#         is_useful = all(
#             dist_to_query < self.dist(self.vectors[cid], self.vectors[s])
#             for s in selected
#         )

#         if not selected or is_useful:
#             selected.append(cid)
#     return selected

## **Insert**

#### We can finally define an insertion function! This uses both *search_layer* and *select_neighbors_heuristic*.

#### Our input includes: our multilayer graph (*hnsw*), the new node to be added (*q*), # of established connections (*M*), max # of connections for each layer per layer (*Mmax*), *efConstruction*, and a normalization factor (*mL*)

1. 

In [ ]:
# import numpy as np

# def insert(self, vector, use_heuristic=True):
    
#     node_id = len(self.vectors) # new node_id is current length of self.vectors
#     self.vectors.append(np.array(vector, dtype=np.float32)) # add vec to self.vectors list
    
#     node_layer = self._sample_layer() # sample layer for new node
    
#     while len(self.graphs) <= node_layer: # make sure self.graphs has enough layers
#         self.graphs.append({}) # adding empty dict for new layer in self.graphs
        
#     if self.entry_point is None: # if first node, set as entry pt, initialize layers
#         self.entry_point = node_id
#         self.max_layer = node_layer
#         for l in range(node_layer+1): # Initialize layers up to node_layer
#             self.graphs[l][node_id] = [] # node_id added to each layer's graph with empty neighbors listed
#         return node_id # return new node_id after insertion
    
#     # So far, we have:
#     # - registered a node at every layer up to its assigned layer
#     # - set it as the entry point
#     # - returned
#     # - Now we must connect to neighbors in each layer up to node_layer! (uses select_neighbors_simple or select_neighbors_heuristic)

#     ep = [self.entry_point]
    
#     # Search from top layer down to node_layer+1 to find entry point for node_layer
#     for l in range(self.max_layer, node_layer, -1):
#         results = self.search_layer(vector, ep, ef=1, layer=l)
#         ep = [results[0][1]] # update entry point for next layer search
    
#     # Now at node_layer, search for neighbors to connect to in layers 0 to node_layer
#     for l in range(min(node_layer, self.max_layer), -1, -1): 
#         results = self.search_layer(vector, ep, ef=self.ef_construction, layer=l)

#         M_layer = self.M0 if l==0 else self.M # M0 for layer 0, M for others
#         if use_heuristic:
#             neighbors = self.select_neighbors_heuristic(vector, results, M_layer)
#         else:
#             neighbors = self.select_neighbors_simple(results, M_layer)

#         self.graphs[l][node_id] = neighbors # connect new node to selected neighbors in layer l

#     # Connect new node to selected neighbors in layer l

#         for nb in neighbors:
#             if nb not in self.graphs[l]:
#                 self.graphs[l][nb] = []
#             self.graphs[l][nb].append(node_id) # connect neighbor to new node

#             # Connect new node to neighbor in layer l
#             if len(self.graphs[l][nb]) > M_layer:
#                 nb_candidates = [
#                     (self.dist(self.vectors[nb], self.vectors[x]), x)
#                     for x in self.graphs[l][nb]
#                 ]
#                 # If heuristic is used, select neighbors based on heuristic
#                 if use_heuristic: 
#                         self.graphs[l][nb] = self.select_neighbors_heuristic(
#                             self.vectors[nb], nb_candidates, M_layer
#                         )
#                 else:
#                     self.graphs[l][nb] = self.select_neighbors_simple(
#                         nb_candidates, M_layer
#                     )
#         ep = [nid for _, nid in results] # update entry point for next layer search
        
#     if node_layer > self.max_layer: # if new node's layer > current max_layer, update max_layer and entry point
#         self.max_layer = node_layer
#         self.entry_point = node_id # update entry point to new node if above is true   

#     return node_id # return new node_id after insertion


## All together...

In [ ]:
print("Hello")

In [ ]:
import numpy as np
import heapq

class HNSW:
    def __init__(self, M=16, ef_construction=200, distance='euclidean', seed=None):
        self.M = M
        self.M0 = 2 * M                    # layer 0 gets more neighbors
        self.ef_construction = ef_construction
        self.mL = 1.0 / np.log(M)         # controls layer assignment spread

        if distance == 'euclidean':
            self.dist = HNSW.euclidean
        elif distance == 'cosine':
            self.dist = HNSW.cosine
        else:
            raise ValueError(f"Unknown distance: {distance}")

        self.rng = np.random.default_rng(seed)

        self.vectors = []        # list of np.ndarray — the actual data
        self.graphs = []         # graphs[layer][node_id] = [neighbor_ids]
        self.entry_point = None  # single entry node at the top layer
        self.max_layer = -1

    # ------------------------------------------------------------------
    # Distance functions
    # ------------------------------------------------------------------

    @staticmethod
    def euclidean(a, b):
        diff = a - b
        return float(np.dot(diff, diff))   # squared L2, no sqrt needed

    @staticmethod
    def cosine(a, b):
        denom = np.linalg.norm(a) * np.linalg.norm(b)
        if denom == 0:
            return 1.0
        return 1.0 - float(np.dot(a, b) / denom)

    # ------------------------------------------------------------------
    # Layer sampling
    # ------------------------------------------------------------------

    def sample_layer(self):
        # exponential distribution — most nodes land at 0,
        # probability roughly halves with each layer up
        return min(int(-np.log(self.rng.uniform()) * self.mL), 16)

    # ------------------------------------------------------------------
    # Core search primitive
    # ------------------------------------------------------------------

    def search_layer(self, query, entry_points, ef, layer):
        visited = set(entry_points)
        candidates = []   # min-heap: (dist, node_id) — what to explore next
        found = []        # max-heap: (-dist, node_id) — best ef results so far

        for ep in entry_points:
            d = self.dist(query, self.vectors[ep])
            heapq.heappush(candidates, (d, ep))
            heapq.heappush(found, (-d, ep))

        while candidates:
            c_dist, c_id = heapq.heappop(candidates)

            worst_found = -found[0][0]
            if c_dist > worst_found:
                break   # can't improve — stop

            for neighbor_id in self.graphs[layer].get(c_id, []):
                if neighbor_id in visited:
                    continue
                visited.add(neighbor_id)

                n_dist = self.dist(query, self.vectors[neighbor_id])
                worst_found = -found[0][0]

                if n_dist < worst_found or len(found) < ef:
                    heapq.heappush(candidates, (n_dist, neighbor_id))
                    heapq.heappush(found, (-n_dist, neighbor_id))
                    if len(found) > ef:
                        heapq.heappop(found)   # evict worst

        return sorted((-d, nid) for d, nid in found)

    # ------------------------------------------------------------------
    # Neighbor selection
    # ------------------------------------------------------------------

    def select_neighbors_simple(self, candidates, M):
        # just take the M closest — candidates is already sorted ascending
        return [nid for _, nid in candidates[:M]]

    def select_neighbors_heuristic(self, query, candidates, M):
        # keep only neighbors that point in genuinely new directions
        selected = []

        for dist_to_query, cid in candidates:
            if len(selected) >= M:
                break

            is_useful = all(
                dist_to_query < self.dist(self.vectors[cid], self.vectors[s])
                for s in selected
            )

            if not selected or is_useful:
                selected.append(cid)

        return selected

    # ------------------------------------------------------------------
    # Insertion
    # ------------------------------------------------------------------

    def insert(self, vector, use_heuristic=True):
        node_id = len(self.vectors)
        self.vectors.append(np.array(vector, dtype=np.float32))

        node_layer = self.sample_layer()

        while len(self.graphs) <= node_layer:
            self.graphs.append({})

        # first node — no search, no edges, just register and return
        if self.entry_point is None:
            self.entry_point = node_id
            self.max_layer = node_layer
            for lc in range(node_layer + 1):
                self.graphs[lc][node_id] = []
            return node_id

        ep = [self.entry_point]

        # phase 1: fast greedy descent from top layer to node_layer+1
        # ef=1 means take only the single best at each layer — just getting close
        for lc in range(self.max_layer, node_layer, -1):
            results = self.search_layer(vector, ep, ef=1, layer=lc)
            ep = [results[0][1]]

        # phase 2: proper beam search from node_layer down to 0
        # this is where we actually find and wire up neighbors
        for lc in range(min(node_layer, self.max_layer), -1, -1):
            results = self.search_layer(vector, ep, ef=self.ef_construction, layer=lc)

            M_layer = self.M0 if lc == 0 else self.M
            if use_heuristic:
                neighbors = self.select_neighbors_heuristic(vector, results, M_layer)
            else:
                neighbors = self.select_neighbors_simple(results, M_layer)

            # connect new node to its selected neighbors
            self.graphs[lc][node_id] = neighbors

            # connect each neighbor back to new node (bidirectional edges)
            for nb in neighbors:
                if nb not in self.graphs[lc]:
                    self.graphs[lc][nb] = []
                self.graphs[lc][nb].append(node_id)

                # if neighbor now exceeds M connections, prune it back down
                if len(self.graphs[lc][nb]) > M_layer:
                    nb_candidates = [
                        (self.dist(self.vectors[nb], self.vectors[x]), x)
                        for x in self.graphs[lc][nb]
                    ]
                    if use_heuristic:
                        self.graphs[lc][nb] = self.select_neighbors_heuristic(
                            self.vectors[nb], nb_candidates, M_layer
                        )
                    else:
                        self.graphs[lc][nb] = self.select_neighbors_simple(
                            nb_candidates, M_layer
                        )

            # carry best candidates down as entry points for next layer
            ep = [nid for _, nid in results]

        # if new node was promoted above current max, it becomes the new entry point
        if node_layer > self.max_layer:
            self.max_layer = node_layer
            self.entry_point = node_id

        return node_id

    # ------------------------------------------------------------------
    # Query
    # ------------------------------------------------------------------

    def query(self, vector, k=10, ef_search=None):
        if self.entry_point is None:
            return []

        if ef_search is None:
            ef_search = max(k, self.ef_construction // 2)
        ef_search = max(ef_search, k)

        vector = np.array(vector, dtype=np.float32)
        ep = [self.entry_point]

        # phase 1: fast descent from top layer to layer 1
        for lc in range(self.max_layer, 0, -1):
            results = self.search_layer(vector, ep, ef=1, layer=lc)
            ep = [results[0][1]]

        # phase 2: full beam search at layer 0
        results = self.search_layer(vector, ep, ef=ef_search, layer=0)

        return results[:k]

    # ------------------------------------------------------------------
    # Diagnostics
    # ------------------------------------------------------------------

    def stats(self):
        n = len(self.vectors)
        layer_counts = [len(g) for g in self.graphs]
        avg_degree = [
            round(float(np.mean([len(nb) for nb in g.values()])), 2) if g else 0
            for g in self.graphs
        ]
        return {
            "n_vectors": n,
            "max_layer": self.max_layer,
            "entry_point": self.entry_point,
            "nodes_per_layer": layer_counts,
            "avg_degree_per_layer": avg_degree,
        }


In [ ]:
# Building a small index
index = HNSW(M=4, ef_construction=20, seed=42)

points = np.random.default_rng(42).uniform(0, 10, size=(20, 2)).astype(np.float32)
for p in points:
    index.insert(p)

print(index.stats())

# Query and compare to brute force
query = np.array([5.0, 5.0], dtype=np.float32)
results = index.query(query, k=3)

print("\nHNSW top 3:", results)

# Brute force ground truth
dists = [(float(np.sum((query - p)**2)), i) for i, p in enumerate(points)]
bf = sorted(dists)[:3]
print("Brute force top 3:", bf)